In [34]:
import pandas as pd

running = pd.read_csv("../data/processed/bigwest_2021_2026_running.csv")
field = pd.read_csv("../data/processed/bigwest_2021_2026_field.csv")

athletes = pd.concat([
    running[["athlete_id", "athlete_name"]],
    field[["athlete_id", "athlete_name"]]
]).drop_duplicates(subset=["athlete_id"])

In [35]:
athlete_ids = (
    athletes["athlete_id"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda x: x.ne("")]
    .unique()
)

In [42]:
url = "https://api.parse.bot/scraper/1b66c7cb-769a-4301-82ff-8d308afee7ad/get_athlete_results"

headers = {
    "X-API-Key": "pmx_e2173f90c664df059780119f36d518e4",
    "API-Snapshot-Version": "5"
}

In [ ]:
import requests
import pandas as pd
import time

# Load existing results
results_df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

# Track completed athletes
completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

# Find athletes still needing data
remaining_ids = [
    str(x).strip()
    for x in athlete_ids
    if str(x).strip() not in completed_ids
]

remaining_ids = list(dict.fromkeys(remaining_ids))

print(f"Starting collection for {len(remaining_ids)} athletes")

new_rows = []

for i, athlete_id in enumerate(remaining_ids):

    try:
        response = requests.get(
            url,
            headers=headers,
            params={"athlete_id": athlete_id},
            timeout=120
        )

        # Stop when rate-limited
        if response.status_code == 429:
            print("Rate limit reached.")
            print("Retry after:", response.headers.get("Retry-After"))
            print("Daily remaining:", response.headers.get(
                "x-ratelimit-daily-remaining"
            ))
            break

        response.raise_for_status()

        result = response.json()

        if result.get("status") != "success":
            print(f"Unsuccessful response for {athlete_id}")
            continue

        athlete = result["data"]

        athlete_rows = []

        for performance in athlete.get("performances", []):

            meet_date = str(performance.get("meet_date", ""))

            # Remove incorrectly parsed records
            if not any(month in meet_date for month in [
                "Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
            ]):
                continue

            athlete_rows.append({
                "athlete_id": athlete_id,
                "athlete_name": athlete.get("name"),
                "team": athlete.get("team"),
                **performance
            })

        if athlete_rows:
            new_rows.extend(athlete_rows)

            # Combine old and new results
            updated_df = pd.concat(
                [results_df, pd.DataFrame(new_rows)],
                ignore_index=True
            )

            updated_df = updated_df.drop_duplicates()

            # Save after each athlete
            updated_df.to_csv(
                "../data/raw/bigwest_regular_season_partial.csv",
                index=False
            )

        completed_ids.add(athlete_id)

        print(
            f"Completed {i + 1}/{len(remaining_ids)}: "
            f"{athlete.get('name')}"
        )

        # Stop near daily quota
        daily_remaining = response.headers.get(
            "x-ratelimit-daily-remaining"
        )

        if daily_remaining is not None and int(daily_remaining) <= 1:
            print("Daily API quota almost exhausted.")
            break

    except (requests.RequestException, ValueError, KeyError, TypeError) as e:
        print(f"Failed {athlete_id}: {e}")

    # Stay within the 5 requests/minute refill rate
    time.sleep(13)

print("Collection stopped. Progress saved.")

Starting collection for 1270 athletes
Completed 1/1270: Chris Shiley
Completed 2/1270: Aiden Davis
Completed 3/1270: Jared Jones
Completed 4/1270: Mateo Joseph
Completed 5/1270: Alex Kalmar
Completed 6/1270: Sean Caster
Completed 7/1270: Dylan Manning
Completed 8/1270: William Fallini-Haas
Completed 9/1270: Talman Young
Completed 10/1270: Brandon Cobian


In [ ]:
results_df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

completed_ids = set(
    results_df["athlete_id"].dropna().astype(str).str.strip()
)

remaining_ids = set(
    str(x).strip() for x in athlete_ids
) - completed_ids

print("Total athletes:", len(athlete_ids))
print("Athletes collected:", len(completed_ids))
print("Athletes remaining:", len(remaining_ids))
print("Total performance records:", len(results_df))

## Cleaning Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/bigwest_regular_season_partial.csv")

# Rename columns
df = df.rename(columns={
    "athete_name": "athlete_name",
    "is_track_field": "is_track"
})

# Clean whitespace
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col].astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

# Preserve original marks for extraction
original_mark = df["mark"].astype("string")

# Extract year, including dates like "Apr 13-16, 2022"
df["meet_year"] = pd.to_numeric(
    df["meet_date"].astype("string").str.extract(r"\b(\d{4})\b")[0],
    errors="coerce"
).astype("Int64")

# Extract meters from marks like 6.72m(0.2)22' 0.75"
meters = pd.to_numeric(
    original_mark.str.extract(r"(\d+(?:\.\d+)?)\s*m\b")[0],
    errors="coerce"
)

# Extract wind
df["wind"] = pd.to_numeric(
    original_mark.str.extract(
        r"^\s*\d+(?:\.\d+)?\s*m?\s*\(\s*([+-]?\d+(?:\.\d+)?)\s*\)"
    )[0],
    errors="coerce"
)

# Read running times in seconds
seconds = pd.to_numeric(df["mark_seconds"], errors="coerce")

# Match event labels without changing the original event column
event = (
    df["event"].astype("string")
    .str.upper()
    .str.replace(r"\s+", "", regex=True)
)

field_event = event.isin([
    "LJ", "TJ", "HJ", "PV", "SP", "DT", "HT", "JT", "WT"
])

running_event = event.isin([
    "60", "100", "200", "400", "800", "1000", "1500",
    "MILE", "3000", "5000", "10000",
    "60H", "100H", "110H", "400H",
    "2000S", "3000S", "4X100", "4X400", "4X800", "DMR","10,000"
])

points_event = event.isin([
    "DEC", "HEP", "PENT", "DECATHLON", "HEPTATHLON", "PENTATHLON","Dec","Hep"
])

# Combine performances into one mark column
df["mark"] = original_mark.str.replace(
    r"\(\s*[+-]?\d+(?:\.\d+)?\s*\)", "", regex=True
).str.strip()
df["mark_unit"] = pd.Series(pd.NA, index=df.index, dtype="string")

df.loc[seconds.notna() | running_event, "mark_unit"] = "time"

df.loc[meters.notna(), "mark"] = meters[meters.notna()].astype("string")
df.loc[meters.notna() | field_event, "mark_unit"] = "meters"

# Combined events use points
points = pd.to_numeric(
    original_mark.str.replace(",", "", regex=False),
    errors="coerce"
)
df.loc[points_event, "mark"] = points[points_event].astype("string")
df.loc[points_event, "mark_unit"] = "points"

# Preserve non-performance statuses
status = original_mark.str.strip().str.upper()
is_status = status.isin(["DNF", "DQ", "DNS", "NM", "NH", "FOUL"])
df.loc[is_status, "mark"] = status[is_status]

# Extract numeric placement
df["place"] = pd.to_numeric(
    df["place"].astype("string").str.extract(r"^(\d+)")[0],
    errors="coerce"
).astype("Int64")

# Select final columns
df = df[[
    "athlete_id",
    "athlete_name",
    "team",
    "meet_name",
    "meet_year",
    "event",
    "mark",
    "mark_unit",
    "place",
    "is_track",
    "wind"
]].copy()

df = df[df["is_track"] == True].copy()

df["event_type"] = "unknown"

df.loc[running_event.reindex(df.index, fill_value=False), "event_type"] = "track"
df.loc[field_event.reindex(df.index, fill_value=False), "event_type"] = "field"
df.loc[points_event.reindex(df.index, fill_value=False), "event_type"] = "combined"
df.head()

In [ ]:
df.to_csv("../data/processed/regular_season.csv", index=False)